# Scraping facility data from datacenters.com

This notebook goes through datacenters.com country by country and pulls one row per data center: who runs it, where it is, how big it is, how much power it uses and when it opened. 
A few things to know before starting:

- **It's slow on purpose.** There's a short pause after every page so we don't hammer the site. All 27 EU countries means a few thousand pages, so plan for several hours.
- **You can stop whenever you like.** Press the stop button (interrupt the kernel). It finishes the page it's on, saves it and stops. Run the scrape cell again later and it carries on from there.
- **Missing values are written as "N/A"**, not left blank, so "not on the page" is easy to tell apart from "something broke".


### Setup & Imports

In [2]:
# !pip install playwright beautifulsoup4 lxml pandas
# !playwright install chromium

In [ ]:
from pathlib import Path

# Countries to scrape. "ALL" means all 27 EU members, or give a list like ["Austria", "Czechia"].
COUNTRIES = ["Malta"]

# Optional: specific datacenters.com location pages, e.g. a single city.
MARKETS = []   # ["https://www.datacenters.com/locations/austria/vienna"]

# Folders. The project folder is the first one upwards that contains .git
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
RAW = ROOT / "data" / "raw" / "datacenters_com"   # progress files: .urls.json and .partial.jsonl
CLEAN = ROOT / "data" / "clean"                   # finished table: .csv and .json
RAW.mkdir(parents=True, exist_ok=True)
CLEAN.mkdir(parents=True, exist_ok=True)

NAME = "eu_datacenters"    # every file this notebook writes starts with this
OUT = str(RAW / NAME)
SHOW_BROWSER = False       # True opens a visible browser window, handy the first time
REFRESH_URLS = False       # True collects the facility list again instead of reusing the saved one
DELAY_SECONDS = 2.0        # pause after each page. Please don't go much lower than this.

In [ ]:
import asyncio
import json
import re
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeout
from contextlib import contextmanager
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright, TimeoutError as PWTimeout

BASE = "https://www.datacenters.com"
NA = "N/A"
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
              "Chrome/124.0 Safari/537.36 research-scraper")

BLOCKED_RESOURCES = {"image", "media", "font"} # no need to download these, saves bandwidth and speeds up scraping

In [ ]:
EU_COUNTRIES = {
    "Austria": (["austria"], ["Austria", "Österreich"]),
    "Belgium": (["belgium"], ["Belgium", "Belgique", "België"]),
    "Bulgaria": (["bulgaria"], ["Bulgaria"]),
    "Croatia": (["croatia"], ["Croatia", "Hrvatska"]),
    "Cyprus": (["cyprus"], ["Cyprus"]),
    "Czechia": (["czech-republic", "czechia"], ["Czech Republic", "Czechia"]),
    "Denmark": (["denmark"], ["Denmark", "Danmark"]),
    "Estonia": (["estonia"], ["Estonia"]),
    "Finland": (["finland"], ["Finland", "Suomi"]),
    "France": (["france"], ["France"]),
    "Germany": (["germany"], ["Germany", "Deutschland"]),
    "Greece": (["greece"], ["Greece"]),
    "Hungary": (["hungary"], ["Hungary", "Magyarország"]),
    "Ireland": (["ireland"], ["Ireland"]),
    "Italy": (["italy"], ["Italy", "Italia"]),
    "Latvia": (["latvia"], ["Latvia"]),
    "Lithuania": (["lithuania"], ["Lithuania"]),
    "Luxembourg": (["luxembourg"], ["Luxembourg"]),
    "Malta": (["malta"], ["Malta"]),
    "Netherlands": (["netherlands", "the-netherlands"], ["Netherlands", "Nederland", "The Netherlands"]),
    "Poland": (["poland"], ["Poland", "Polska"]),
    "Portugal": (["portugal"], ["Portugal"]),
    "Romania": (["romania"], ["Romania"]),
    "Slovakia": (["slovakia"], ["Slovakia", "Slovak Republic"]),
    "Slovenia": (["slovenia"], ["Slovenia"]),
    "Spain": (["spain"], ["Spain", "España"]),
    "Sweden": (["sweden"], ["Sweden", "Sverige"]),
}

FIELDS = [
    "country",
    "type_of_data_center",
    "name",
    "owner",
    "land_use",
    "building_sqft",
    "colocation_sqft",
    "power_mw",
    "energy_consumption_electricity",
    "energy_consumption_water",
    "pue",
    "location",
    "city",
    "state",
    "postal_code",
    "latitude",
    "longitude",
    "source_of_electricity",
    "certifications",
    "year_built_or_operational",
    "year_evidence",
    "market",        # country/state/city as datacenters.com files it, e.g. germany/bayern/munchen
    "source_url",
]

In [ ]:
NON_FACILITY = {
    "locations", "providers", "services", "news", "blog", "events", "about", "contact", "login",
    "signup", "register", "search", "privacy", "terms", "cookies", "colocation", "cloud", "bare-metal",
    "hyperscale", "edge", "marketplace", "partners", "resources", "careers", "faq", "sitemap",
    "data-center-locations", "pricing", "quote", "api", "assets", "images", "static", "cdn-cgi",
    "about-us", "contact-us", "terms-of-use", "privacy-policy", "cookie-policy", "rfp-templates",
}
FACILITY_PATH = re.compile(r"^/[a-z0-9]+(?:-[a-z0-9]+)+/?$")       # a single hyphenated slug
MARKET_PATH = re.compile(r"^/locations/[^/]+(?:/[^/]+){0,2}/?$")

TYPE_WORDS = r"(Colocation|Hyperscale|Edge|Enterprise|Wholesale|Cloud|Carrier Hotel|Build[- ]to[- ]Suit|" \
             r"Powered Shell|Disaster Recovery|Interconnection|Modular|HPC|AI)"

# Years: look for phrases like "opened in 2014" or "Year built: 2009"
YEAR = r"((?:19[6-9]\d|20[0-4]\d))"
YEAR_PATTERNS = [
    rf"(?:year built|year opened|built|opened|completed|commissioned|operational since|in operation since|"
    rf"live since|established|launched|ready for service|RFS)\s*[:\-]\s*(?:\w+\s+)?{YEAR}",
    rf"(?:was |were |is |has been |first )?(?:built|constructed|completed|opened|commissioned|launched|"
    rf"established|inaugurated|went live|came online|entered (?:into )?service|began operations?|"
    rf"started operations?|operational|in operation|in service|live)"
    rf"\s+(?:in|since|from|during)\s+(?:(?:early|mid|late)[- ])?(?:Q[1-4]\s+|"
    rf"(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+)?{YEAR}",
    rf"(?:opened|launched|commissioned|completed)\s+(?:its|the|this)\s+[\w\s-]{{0,30}}?\s+in\s+{YEAR}",
    rf"since (?:its )?(?:opening|launch|construction|completion|commissioning)\s+in\s+{YEAR}",
    rf"{YEAR}[- ](?:built|build|vintage)",
]
# Lines with these words tend to have years that have nothing to do with construction
YEAR_NOISE = re.compile(r"as of|partner of the year|award|price|tariff|VAT|copyright|©|blog|podcast|"
                        r"news|acquired|acquisition|risk class|cloud region", re.I)

# Keys in the page's embedded data that might hold an opening year 
YEAR_KEY = re.compile(r"year|open|built|operational|commission|rfs|launch|go_?live", re.I)
YEAR_KEY_NOISE = re.compile(r"creat|updat|modif|publish|price", re.I)

PUE_VALUE = re.compile(r"PUE\s*(?:of|:|=)?\s*(?:[<≤]=?\s*|less than\s*|below\s*|under\s*)?(\d\.\d{1,2})", re.I)

CERT_PATTERNS = [
    r"ISO[/ ]?(?:IEC )?\d{4,5}(?::\d{4})?",
    r"SOC ?[123](?: Type (?:II|I|1|2))?",
    r"PCI[- ]DSS",
    r"HIPAA",
    r"TISAX",
    r"BSI C5|C5 attest\w*",
    r"EN ?50600(?:[- ][\w.-]+)?",
    r"Tier (?:IV|III|II|I|[1-4])\b",
    r"Uptime Institute",
    r"LEED (?:Platinum|Gold|Silver|Certified)",
    r"BREEAM (?:Outstanding|Excellent|Very Good|Good)",
]

# Postcodes in the formats EU countries actually use (NL, PL/PT, MT, then plain digits)
POSTCODE = [r"\b\d{4} ?[A-Z]{2}\b", r"\b\d{2}-\d{3}\b|\b\d{4}-\d{3}\b", r"\b[A-Z]{3} ?\d{4}\b", r"\b\d{4,5}\b"]

In [ ]:
def clean(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def first_match(patterns, text):
    # Despite the name, this collects every distinct hit, joined with "; "
    out, seen = [], set()
    for p in patterns:
        for m in re.finditer(p, text, flags=re.I):
            hit = clean(m.group(0)).rstrip(".,;")
            # skip exact repeats, and bare numbers we already have with a label in front
            if hit.lower() not in seen and not any(hit.lower() in s for s in seen):
                seen.add(hit.lower())
                out.append(hit)
    return "; ".join(out) if out else NA


def walk(obj):
    # every dict inside a nested JSON structure
    if isinstance(obj, dict):
        yield obj
        for v in obj.values():
            yield from walk(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from walk(v)


def pick(d, *keys):
    # first non-empty value among keys, ignoring upper/lower case
    lower = {k.lower(): v for k, v in d.items()} if isinstance(d, dict) else {}
    for k in keys:
        v = lower.get(k.lower())
        if v not in (None, "", [], {}):
            return v
    return None


def as_name(v):
    # some fields are plain strings, others are {"name": ...}
    if isinstance(v, dict):
        v = v.get("name")
    return clean(v) if isinstance(v, str) and v.strip() else None


def country_from_url(url):
    # display name for a /locations/<country>/... URL
    parts = urlparse(url).path.strip("/").split("/")
    if len(parts) > 1 and parts[0] == "locations":
        slug = parts[1]
        for name, (slugs, _) in EU_COUNTRIES.items():
            if slug in slugs:
                return name
        return slug.replace("-", " ").title()
    return None


def address_names(country):
    return EU_COUNTRIES.get(country, ([], [country] if country else []))[1]

### Browser Setup

In [ ]:
_robots = None


def load_robots(context):
    # Fetch robots.txt with the browser's own request context, so we get the same answer a visitor gets
    global _robots
    rp = RobotFileParser()
    try:
        resp = context.request.get(BASE + "/robots.txt", timeout=20000)
        rp.parse(resp.text().splitlines() if resp.ok else [])   # no file = no rules
    except Exception as e:
        print(f"Couldn't read robots.txt ({e}), carrying on without it")
        rp.parse([])
    _robots = rp


def allowed(url):
    return _robots is None or _robots.can_fetch(USER_AGENT, url)


@contextmanager
def open_browser(show=False):
    with sync_playwright() as pw:
        browser = pw.chromium.launch(headless=not show)
        context = browser.new_context(user_agent=USER_AGENT, locale="en-US")
        context.route("**/*", lambda route: route.abort()
                      if route.request.resource_type in BLOCKED_RESOURCES else route.continue_())
        load_robots(context)
        try:
            yield context.new_page()
        finally:
            browser.close()


def open_page(page, url):
    # Returns the final URL once loaded, False if the page is gone (404/410)
    # or not allowed, and None if it just wouldn't load (we'll try again next run).
    if not allowed(url):
        print(f"   [skip] robots.txt says no: {url}")
        return False
    for attempt in range(3):
        try:
            resp = page.goto(url, wait_until="domcontentloaded", timeout=45000)
            if resp is not None and resp.status in (404, 410):
                print(f"   [gone] {resp.status}: {url}")
                return False
            try:
                page.wait_for_load_state("networkidle", timeout=15000)
            except PWTimeout:
                pass   # some pages never go fully quiet, that's fine
            time.sleep(DELAY_SECONDS)
            return page.url
        except Exception as e:
            print(f"   [retry {attempt + 1}] {url}: {e}")
            time.sleep(5 * (attempt + 1))
    return None


def current_links(page):
    try:
        return set(page.eval_on_selector_all("a[href]", "els => els.map(e => e.href)"))
    except Exception:
        return set()   # happens if the page is halfway through navigating


def is_facility_url(url):
    p = urlparse(url)
    if not p.netloc.endswith("datacenters.com"):
        return False
    first = p.path.strip("/").split("/")[0]
    return bool(FACILITY_PATH.match(p.path)) and first not in NON_FACILITY


def facility_links(page):
    return {u for u in current_links(page) if is_facility_url(u)}


def expand_listing(page, start_path, max_rounds=300):
    # Country listings show about 48 facilities at a time with a "Next page" button that
    # swaps the list without changing the URL. Links are collected after every click.
    seen = current_links(page)
    for _ in range(max_rounds):
        before = len(seen)
        page.mouse.wheel(0, 20000)
        page.wait_for_timeout(500)
        shown = facility_links(page)
        clicked = False
        for label in ("Next page", "Load more", "Show more", "View more", "More locations", "Next"):
            btn = page.get_by_role("button", name=re.compile(rf"^{label}", re.I))
            if btn.count() == 0:
                btn = page.get_by_role("link", name=re.compile(rf"^{label}$", re.I))
            if btn.count() > 0 and btn.first.is_visible() and btn.first.is_enabled():
                try:
                    btn.first.click(timeout=3000)
                    clicked = True
                    break
                except Exception:
                    pass
        if clicked:
            # wait until the list really changes (up to about 10 seconds)
            for _ in range(20):
                page.wait_for_timeout(500)
                if facility_links(page) - shown:
                    break
        # if that click took us somewhere outside this listing, go back and stop
        if not urlparse(page.url).path.rstrip("/").startswith(start_path):
            page.go_back()
            break
        seen |= current_links(page)
        if len(seen) == before:
            break   # nothing new appeared, so this was the last page
    return seen

In [ ]:
stop_flag = threading.Event()


def run_in_background(fn, *args, **kwargs):
    stop_flag.clear()

    def worker():
        if sys.platform == "win32":
            # Jupyter on Windows can leave a loop type that can't start the browser
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        return fn(*args, **kwargs)

    with ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(worker)
        while True:
            try:
                return future.result(timeout=0.5)
            except FutureTimeout:
                continue
            except KeyboardInterrupt:
                if not stop_flag.is_set():
                    print("\nStopping after the current page, hang on...")
                    stop_flag.set()

### Finding Facilities

Start from a country page (like `/locations/austria`), collect every facility link and step into the city pages below it.

In [ ]:
def collect_facility_urls(page, start_url):
    final_url = open_page(page, start_url)
    if not final_url:
        return {}
    final_url = final_url.split("?")[0].rstrip("/")
    start_path = urlparse(final_url).path.rstrip("/")
    market = start_path.replace("/locations/", "", 1)

    urls = {urljoin(BASE, u) for u in rsc_listing_urls(rsc_payload(page.content()))}
    urls |= {u.split("?")[0].split("#")[0].rstrip("/") for u in expand_listing(page, start_path)}
    found = {u: market for u in urls if is_facility_url(u)}
    print(f"  {len(found)} facility links on {final_url}")
    return found

### Running browser from Jupyter


In [ ]:
# The site is built with Next.js. Each page carries its data as "React Server Component"
# chunks inside <script>self.__next_f.push([1,"..."])</script> tags. On a facility page,
# one chunk holds a "location" object with the address, coordinates, size, power,
# operator, facility types and certifications. That is the main source from here on.
NEXT_PUSH = re.compile(r'self\.__next_f\.push\(\[1,"((?:[^"\\]|\\.)*)"\]\)')


def rsc_payload(html):
    parts = []
    for chunk in NEXT_PUSH.findall(html):
        try:
            parts.append(json.loads(f'"{chunk}"'))
        except json.JSONDecodeError:
            pass
    return "".join(parts)


def rsc_text(rsc, ref):
    # Long texts are stored separately as rows like '2e:T150c,<p>...' (length in bytes, hex)
    m = re.search(r"(?:^|\n|>)" + re.escape(ref) + r":T([0-9a-f]+),", rsc)
    if not m:
        return ""
    return rsc[m.end():].encode("utf-8")[:int(m.group(1), 16)].decode("utf-8", "ignore")


def rsc_location(rsc, slug):
    # The facility's "location" object. Nearby facilities have smaller objects without an
    # address, so the one matching this page's slug (or holding an address) is taken.
    decoder, best = json.JSONDecoder(), None
    for m in re.finditer(r'"location":\s*\{', rsc):
        try:
            obj, _ = decoder.raw_decode(rsc, m.end() - 1)
        except json.JSONDecodeError:
            continue
        if "fullAddress" not in obj and "provider" not in obj:
            continue
        if (obj.get("locationProvider") or {}).get("slug") == slug:
            return obj
        best = best or obj
    return best


def rsc_listing_urls(rsc):
    # Facility list on a country page: {"name": ..., "url": "/northc-nuremberg", "fullAddress": ...}
    return list(dict.fromkeys(re.findall(r'"url":\s*"(/[a-z0-9-]+)",\s*"fullAddress"', rsc)))


def json_blobs(soup):
    # every JSON chunk on the page: JSON-LD, Next.js data, React props and similar
    blobs = []
    for tag in soup.find_all("script"):
        kind = (tag.get("type") or "").lower()
        if "json" in kind or tag.get("id") == "__NEXT_DATA__":
            try:
                blobs.append(json.loads(tag.string or ""))
            except (json.JSONDecodeError, TypeError):
                pass
    for attr in ("data-react-props", "data-props", "data-page"):
        for tag in soup.find_all(attrs={attr: True}):
            try:
                blobs.append(json.loads(tag[attr]))
            except (json.JSONDecodeError, TypeError):
                pass
    return blobs


def ld_place(blobs, slug):
    # The JSON-LD object describing this place. If there are several (nearby
    # facilities, for example), we prefer the one whose URL matches the page.
    best, best_score = None, -1
    for blob in blobs:
        for d in walk(blob):
            if "@type" in d and ("address" in d or "geo" in d):
                score = len(d) + (100 if slug in str(d.get("url", "")) else 0)
                if score > best_score:
                    best, best_score = d, score
    if not best:
        return {}

    out = {"name": as_name(best.get("name"))}
    addr = best.get("address")
    if isinstance(addr, list):
        addr = addr[0] if addr else None
    if isinstance(addr, dict):
        out["street"] = clean(addr.get("streetAddress")) or None
        out["postal_code"] = clean(addr.get("postalCode")) or None
        out["city"] = clean(addr.get("addressLocality")) or None
        country = as_name(addr.get("addressCountry")) or clean(addr.get("addressCountry"))
        parts = [out["street"], " ".join(x for x in (out["postal_code"], out["city"]) if x), country]
        out["location"] = ", ".join(x for x in parts if x) or None
    elif isinstance(addr, str):
        out["location"] = clean(addr)

    geo = best.get("geo") if isinstance(best.get("geo"), dict) else {}
    out["lat"], out["lon"] = geo.get("latitude"), geo.get("longitude")

    for key in ("brand", "parentOrganization", "provider", "operator", "owner"):
        owner = as_name(best.get(key))
        if owner and "datacenters.com" not in owner.lower():
            out["owner"] = owner
            break

    out["years"] = []
    for key in ("foundingDate", "dateCreated", "yearBuilt", "openingDate"):
        m = re.search(YEAR, str(best.get(key, "")))
        if m:
            out["years"].append((int(m.group(1)), f"JSON-LD {key}: {best[key]}"))
    return out


def page_entity(blobs, slug):
    # The facility's own record in the app data: a dict whose slug or URL ends with ours
    best, best_n = {}, 0
    for blob in blobs:
        for d in walk(blob):
            for k in ("slug", "url", "path", "permalink", "href", "canonical_url", "canonicalUrl"):
                v = d.get(k)
                if isinstance(v, str) and v.rstrip("/").split("/")[-1] == slug:
                    n = sum(1 for x in d.values() if not isinstance(x, (dict, list)))
                    if n > best_n:
                        best, best_n = d, n
                    break
    return best


def entity_lines(entity):
    # Turn {"totalPower": "10 MW"} into "total power: 10 MW" so the text patterns pick it up
    out = []
    for k, v in entity.items():
        if isinstance(v, bool) or not isinstance(v, (str, int, float)) or not str(v).strip():
            continue
        if isinstance(v, str) and "<" in v:
            v = BeautifulSoup(v, "lxml").get_text(" ")
        label = re.sub(r"(?<!^)(?=[A-Z])", " ", k).replace("_", " ").lower()
        out.append(clean(f"{label}: {v}")[:500])
    return out


def entity_years(entity):
    found = []
    for k, v in entity.items():
        if YEAR_KEY.search(k) and not YEAR_KEY_NOISE.search(k) and isinstance(v, (str, int)):
            m = re.search(YEAR, str(v))
            if m:
                found.append((int(m.group(1)), f"page data {k}: {v}"))
    return found


def find_coords(soup, ld, entity):
    options = [(ld.get("lat"), ld.get("lon")),
               (pick(entity, "latitude", "lat"), pick(entity, "longitude", "lng", "lon", "long"))]
    for a, b in (("place:location:latitude", "place:location:longitude"), ("og:latitude", "og:longitude")):
        ta, tb = soup.find("meta", property=a), soup.find("meta", property=b)
        if ta and tb:
            options.append((ta.get("content"), tb.get("content")))
    tag = soup.find(attrs={"data-lat": True})
    if tag:
        options.append((tag.get("data-lat"), tag.get("data-lng") or tag.get("data-lon") or tag.get("data-long")))
    # Google Maps / OpenStreetMap links and iframes often have the coordinates in the URL
    for t in soup.find_all(["a", "iframe"]):
        src = t.get("href") or t.get("src") or ""
        if "map" in src.lower():
            m = re.search(r"[@=/,](-?\d{1,2}\.\d{3,})(?:,|%2C)\s*(-?\d{1,3}\.\d{3,})", src)
            if m:
                options.append(m.groups())

    for lat, lon in options:
        try:
            lat, lon = float(lat), float(lon)
        except (TypeError, ValueError):
            continue
        if -90 <= lat <= 90 and -180 <= lon <= 180 and (lat, lon) != (0, 0):
            return f"{lat:.6f}", f"{lon:.6f}"
    return NA, NA


def meta_text(soup):
    bits = []
    for attrs in ({"name": "description"}, {"property": "og:description"}):
        t = soup.find("meta", attrs=attrs)
        if t and t.get("content"):
            bits.append(clean(t["content"]))
    return " ".join(dict.fromkeys(bits))

### Parsing Facility Page

This puts everything together into one record. For each field it tries the structured data first and falls back to the text patterns. `parse_html` only needs the HTML, so you can also run it on a page you saved earlier.

In [ ]:
def extract_year(lines, extra=()):
    # We keep the earliest year we find. Later years are usually expansions.
    candidates = list(extra)
    text = " ".join(l for l in lines if not YEAR_NOISE.search(l))
    for p in YEAR_PATTERNS:
        for m in re.finditer(p, text, flags=re.I):
            candidates.append((int(m.group(m.lastindex)), clean(m.group(0))))
    if not candidates:
        return NA, NA
    year = min(y for y, _ in candidates)
    return str(year), "; ".join(dict.fromkeys(e for y, e in candidates if y == year))


def normalise_cert(c):
    # "ISO/IEC 27001:2013" -> "ISO 27001", "PCI-DSS" -> "PCI DSS", so duplicates collapse
    c = re.sub(r"ISO/IEC\s*", "ISO ", clean(c), flags=re.I)
    c = re.sub(r"(ISO \d+):\d{4}", r"\1", c)
    return re.sub(r"PCI[- ]DSS", "PCI DSS", c, flags=re.I)


def parse_html(html, url, final_url=None, country=None, market=None):
    rec = {k: NA for k in FIELDS}
    rec["source_url"] = final_url or url
    rec["country"] = country or NA
    rec["market"] = market or NA

    soup = BeautifulSoup(html, "lxml")
    slug = urlparse(url).path.strip("/")

    # 1. The page's own data object (see "Reading Structured Data")
    rsc = rsc_payload(html)
    loc = rsc_location(rsc, slug) or {}
    rec["_facility"] = bool(loc)
    description = loc.get("description") or ""
    if isinstance(description, str) and description.startswith("$"):
        description = rsc_text(rsc, description[1:])
    description = clean(BeautifulSoup(description, "lxml").get_text(" ")) if description else ""

    def number(key):
        try:
            v = float(loc.get(key) or 0)
        except (TypeError, ValueError):
            return None
        return v if v > 0 else None   # the site uses 0 for "not provided"

    if loc:
        rec["name"] = clean(loc.get("name")) or NA
        rec["owner"] = as_name(loc.get("provider")) or NA
        rec["country"] = clean(loc.get("country")) or rec["country"]
        rec["location"] = clean(loc.get("fullAddress")) or NA
        rec["city"] = clean(loc.get("city")) or NA
        rec["state"] = clean(loc.get("state")) or NA
        rec["postal_code"] = clean(loc.get("zip")) or NA
        if loc.get("latitude") not in (None, "") and loc.get("longitude") not in (None, ""):
            rec["latitude"], rec["longitude"] = f"{float(loc['latitude']):.6f}", f"{float(loc['longitude']):.6f}"
        demo = loc.get("locationDemographic") or {}
        path = [(demo.get(k) or {}).get("slug") for k in ("country", "state", "city")]
        if all(path):
            rec["market"] = "/".join(path)
        types = [((e.get("node") or {}).get("dataCenterFacility") or {}).get("title")
                 for e in (loc.get("publishedDataCenterFacilityLocations") or {}).get("edges", [])]
        rec["type_of_data_center"] = "; ".join(t for t in types if t) or NA
        site_certs = [(e.get("node") or {}).get("abbreviation")
                      for e in (loc.get("publishedDataCenterCertifications") or {}).get("edges", [])]
        building, coloc, power = number("grossBuildingSize"), number("grossColocationSpace"), number("totalPowerMw")
        rec["building_sqft"] = f"{building:.0f}" if building else NA
        rec["colocation_sqft"] = f"{coloc:.0f}" if coloc else NA
        rec["power_mw"] = f"{power:g}" if power else NA
    else:
        site_certs, building, coloc, power = [], None, None, None

    # structured bits first, before the scripts get thrown away
    blobs = json_blobs(soup)
    ld = ld_place(blobs, slug)
    entity = page_entity(blobs, slug)
    if rec["latitude"] == NA:
        rec["latitude"], rec["longitude"] = find_coords(soup, ld, entity)
    meta = meta_text(soup)

    for s in soup(["script", "style", "noscript"]):
        s.extract()
    lines = [clean(l) for l in soup.get_text("\n").split("\n")]
    lines = [l for l in lines if l]
    extra = entity_lines(entity) + ([description] if description else [])
    text = " ".join(lines + extra + [meta])

    # Name and owner. Headings usually look like "NTT Global Data Centers: Vienna 1 Data Center"
    heading = clean(soup.find("h1").get_text()) if soup.find("h1") else ""
    title = clean(soup.title.get_text()) if soup.title else ""
    for cand in (heading, title):
        if rec["name"] != NA and rec["owner"] != NA:
            break
        if ":" in cand:
            owner, name = cand.split(":", 1)
            rec["owner"], rec["name"] = clean(owner), clean(name)
            break
    if rec["name"] == NA:
        rec["name"] = heading or ld.get("name") or as_name(pick(entity, "name", "title")) or NA
    if rec["owner"] == NA:
        rec["owner"] = (ld.get("owner")
                        or as_name(pick(entity, "provider", "providerName", "provider_name", "company",
                                        "companyName", "operator", "owner"))
                        or NA)
    if rec["owner"] == NA:
        m = re.search(r"Contact\s+(.{2,60}?)(?:\s{2,}|$|\n)", "\n".join(lines))
        if m:
            rec["owner"] = clean(m.group(1))
    if rec["owner"] == NA:
        # a link to the provider's profile page, e.g. /providers/equinix
        for a in soup.find_all("a", href=re.compile(r"^(?:https?://[^/]+)?/providers/[^/?#]+/?$")):
            t = clean(a.get_text())
            if 2 <= len(t) <= 60 and not re.search(r"provider|view|all|more", t, re.I):
                rec["owner"] = t
                break
    if rec["owner"] == NA:
        m = re.search(r"(?:Operated|Owned) by\s+([A-Z][\w&.,' -]{1,60})", text)
        if m:
            rec["owner"] = clean(m.group(1))

    # Type, from section headings like "Colocation" or "Hyperscale", or from the text
    if rec["type_of_data_center"] == NA:
        types = []
        for h in soup.find_all(["h2", "h3", "h4"]):
            t = clean(h.get_text())
            if re.fullmatch(TYPE_WORDS + r"(?: Data Cent(?:er|re)s?| Services?)?", t) and t not in types:
                types.append(t)
        rec["type_of_data_center"] = "; ".join(types) if types else NA

    # Location. A structured address wins if it has a street number or postcode,
    # otherwise we look for a short line that mentions the country and has a number in it.
    if rec["location"] == NA and ld.get("location") and re.search(r"\d", ld["location"]):
        rec["location"] = ld["location"]
    names = address_names(country)
    if rec["location"] == NA and names:
        pat = re.compile(r"\b(?:" + "|".join(re.escape(n) for n in names) + r")\b", re.I)
        for l in lines:
            if pat.search(l) and re.search(r"\d", l) and len(l) < 140:
                rec["location"] = l
                break
    if rec["location"] == NA:
        addr = soup.find("address")
        if addr:
            rec["location"] = clean(addr.get_text(" "))
    if rec["location"] == NA:
        rec["location"] = ld.get("location") or clean(pick(entity, "address", "full_address", "fullAddress")) or NA

    # City and postcode, falling back to the city page we found the facility on
    if rec["city"] == NA:
        rec["city"] = ld.get("city") or as_name(pick(entity, "city", "addressLocality", "locality")) or NA
    if rec["postal_code"] == NA:
        rec["postal_code"] = ld.get("postal_code") or clean(pick(entity, "postal_code", "postalCode", "zip", "zipcode")) or NA
    if rec["postal_code"] == NA and rec["location"] != NA:
        for p in POSTCODE:
            m = re.search(p, rec["location"])
            if m:
                rec["postal_code"] = m.group(0)
                break

    # Land use: total / colocation / raised floor / plot areas
    land_text = first_match([
        r"(?:total|colocation|raised floor|white|building|gross|plot|land|site|campus|office)\s+"
        r"(?:space|area|size|footprint)?\s*:?\s*[\d.,]+\s*(?:sq\.?\s?ft|sqft|square feet|m²|m2|sqm|"
        r"square met(?:er|re)s?|acres?|hectares?|ha)\b",
        r"[\d.,]+\s*(?:sq\.?\s?ft|sqft|square feet|m²|m2|sqm|square met(?:er|re)s?|acres?|hectares?)\b"
        r"(?:\s+(?:of\s+)?(?:plot|site|land|campus|data ?cent(?:er|re) space|white ?space|server space|"
        r"office(?: and conference)? space|building|facility|raised floor))?",
    ], text)
    land = []
    if building:
        land.append(f"building {building:,.0f} sqft")
    if coloc:
        land.append(f"colocation {coloc:,.0f} sqft")
    if land_text != NA:
        land.append(land_text)
    rec["land_use"] = "; ".join(land) if land else NA

    # Electricity: capacity, IT load, PUE, yearly consumption
    power_text = first_match([
        r"(?:total |utility |critical |IT )?power(?: capacity)?\s*:?\s*[\d.,]+\s*(?:MW|kW|MVA)\b",
        r"[\d.,]+\s*(?:MW|kW|MVA|GWh|MWh)\b(?:\s+(?:of\s+)?(?:IT load|critical (?:IT )?load|power|capacity|"
        r"future capacity|maximum IT load))?",
        r"PUE\s*(?:of|:)?\s*(?:[<≤]=?\s*|less than\s*|below\s*)?\d+(?:\.\d+)?",
        r"\d+(?:\.\d+)?\s*PUE",
    ], text)
    power_parts = ([f"{power:g} MW total power"] if power else []) + ([power_text] if power_text != NA else [])
    rec["energy_consumption_electricity"] = "; ".join(power_parts) if power_parts else NA
    pues = list(dict.fromkeys(PUE_VALUE.findall(text)))
    rec["pue"] = "; ".join(pues) if pues else NA

    # Water
    rec["energy_consumption_water"] = first_match([
        r"WUE\s*(?:of|:)?\s*\d+(?:\.\d+)?\s*(?:L/kWh)?",
        r"[\d.,]+\s*(?:litres|liters|m³|m3|cubic met(?:er|re)s|gallons)\b(?:\s+of\s+water)?",
        r"(?:water[- ]free|zero[- ]water|closed[- ]loop|adiabatic|evaporative|free|air|liquid) cooling",
    ], text)

    # Source of electricity. These pages only have facility-level claims, so take them with a pinch of salt.
    rec["source_of_electricity"] = first_match([
        r"\d{1,3}\s*%\s*(?:renewable|green|hydro\w*|wind|solar|carbon[- ]free)\s*(?:energy|power|electricity)?",
        r"(?:100%\s*)?(?:renewable|green|hydro(?:electric)?|solar|wind|carbon[- ]neutral|carbon[- ]free)"
        r"[- ](?:energy|power|electricity)(?:\s+(?:supply|sourced|PPA))?",
        r"\bPPA\b[^.]{0,80}",
        r"\b(?:on-?site|onsite)\s+(?:solar|generation|CHP|gas turbines?)\b",
    ], text)

    text_certs = first_match(CERT_PATTERNS, text)
    certs = site_certs + ([] if text_certs == NA else text_certs.split("; "))
    rec["certifications"] = "; ".join(dict.fromkeys(normalise_cert(c) for c in certs if c)) or NA

    rec["year_built_or_operational"], rec["year_evidence"] = extract_year(
        lines + extra + [meta], ld.get("years", []) + entity_years(entity))
    return rec


def parse_facility(page, url, country=None, market=None):
    # None means "couldn't load, try again later". A dict with _skip=True means "not worth retrying".
    final_url = open_page(page, url)
    if final_url is None:
        return None
    if final_url is False:
        return {**{k: NA for k in FIELDS}, "source_url": url, "_skip": True}
    rec = parse_html(page.content(), url, final_url, country, market)
    rec["_skip"] = not rec.pop("_facility")   # no facility data on the page: about, terms, etc.
    return rec

### Saving Process

In [ ]:
def load_jsonl(path):
    try:
        with open(path, encoding="utf-8") as f:
            return [json.loads(l) for l in f if l.strip()]
    except FileNotFoundError:
        return []


def append_jsonl(path, row):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def build_outputs(out=None):
    # JSON + CSV from everything collected so far. If a facility shows up twice, the latest version wins.
    out = out or OUT
    latest = {}
    for r in load_jsonl(f"{out}.partial.jsonl"):
        latest[r.get("_url", r.get("source_url"))] = r
    records = [{k: r.get(k, NA) for k in FIELDS} for r in latest.values() if not r.get("_skip")]

    name = Path(out).name
    json_file, csv_file = CLEAN / f"{name}.json", CLEAN / f"{name}.csv"
    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    df = pd.DataFrame(records, columns=FIELDS)
    df.to_csv(csv_file, index=False, encoding="utf-8-sig")   # utf-8-sig so Excel shows umlauts properly
    print(f"Saved {len(records)} records to {csv_file.relative_to(ROOT)} and {json_file.name}")
    return df

### Main Loop

In [ ]:
def scrape(countries=(), markets=(), out=None, refresh_urls=False, show=False):
    out = out or OUT
    if countries == "ALL":
        countries = list(EU_COUNTRIES)
    unknown = [c for c in countries if c not in EU_COUNTRIES]
    if unknown:
        raise ValueError(f"Unknown country {unknown}. Pick from: {', '.join(EU_COUNTRIES)}")

    urls_file, partial_file = f"{out}.urls.json", f"{out}.partial.jsonl"

    with open_browser(show) as page:
        # 1. Facility list: url -> {"country": ..., "market": ...}
        jobs = {}
        if not refresh_urls:
            try:
                with open(urls_file, encoding="utf-8") as f:
                    jobs = json.load(f)
                # lists saved by the old script were just url -> country
                jobs = {u: v if isinstance(v, dict) else {"country": v, "market": None} for u, v in jobs.items()}
                print(f"Using the saved facility list: {len(jobs)} facilities ({Path(urls_file).relative_to(ROOT)})")
            except FileNotFoundError:
                pass

        if not jobs:
            for country in countries:
                found = {}
                for slug in EU_COUNTRIES[country][0]:   # try the other slug if the first one is empty
                    print(f"Collecting {country} from {BASE}/locations/{slug}")
                    found = collect_facility_urls(page, f"{BASE}/locations/{slug}")
                    if found or stop_flag.is_set():
                        break
                print(f"=> {country}: {len(found)} facilities")
                for u, market in found.items():
                    jobs.setdefault(u, {"country": country, "market": market})
            for m in markets:
                print(f"Collecting from {m}")
                for u, market in collect_facility_urls(page, m).items():
                    jobs.setdefault(u, {"country": country_from_url(m), "market": market})
            if stop_flag.is_set():
                print("Stopped while collecting links. The list isn't complete, so it wasn't saved.")
                return None
            with open(urls_file, "w", encoding="utf-8") as f:
                json.dump(jobs, f, ensure_ascii=False, indent=1)
            print(f"{len(jobs)} facilities in total, list saved to {Path(urls_file).relative_to(ROOT)}")

        # 2. Facility pages, skipping whatever an earlier run already finished
        done = {r["_url"] for r in load_jsonl(partial_file)}
        todo = sorted(((u, j) for u, j in jobs.items() if u not in done),
                      key=lambda x: (x[1].get("country") or "", x[0]))
        print(f"{len(done)} already done, {len(todo)} to go (roughly {len(todo) * 6 / 3600:.1f} hours)")

        failed = 0
        for i, (u, job) in enumerate(todo, 1):
            if stop_flag.is_set():
                print("Stopped. Run the cell again to continue.")
                break
            print(f"[{i}/{len(todo)}] {job.get('country')}: {u}")
            rec = parse_facility(page, u, job.get("country"), job.get("market"))
            if rec is None:
                failed += 1
                print("   (didn't load, will try again next run)")
                continue
            if rec["_skip"]:
                print("   (doesn't look like a facility page, skipped)")
            rec["_url"] = u
            append_jsonl(partial_file, rec)
        if failed:
            print(f"{failed} pages didn't load. Run again to retry them.")

    return build_outputs(out)

In [ ]:
## Check
TEST_URL = "https://www.datacenters.com/ntt-global-vienna-1"


def test_one(url):
    with open_browser(SHOW_BROWSER) as page:
        final_url = open_page(page, url)
        if not final_url:
            print("Couldn't load that page")
            return None
        rec = parse_html(page.content(), url, final_url)
        rec.pop("_facility", None)
        return rec


rec = run_in_background(test_one, TEST_URL)
pd.Series(rec) if rec else None

country                                                                     Austria
type_of_data_center                                                      Colocation
name                                                           Vienna 1 Data Center
owner                                                       NTT Global Data Centers
land_use                          total space 94722 sqft; 8,800m² of server spac...
energy_consumption_electricity                    15.0 MW; 15.0 MW of power; 15.2MW
energy_consumption_water                                                        N/A
pue                                                                             N/A
location                                     Computerstraße 4, 1100 Vienna, Austria
city                                                                            N/A
postal_code                                                                    1100
latitude                                                                    

In [ ]:
df = run_in_background(scrape, COUNTRIES, MARKETS, out=OUT, refresh_urls=REFRESH_URLS, show=SHOW_BROWSER)
df.head() if df is not None else None

Using the saved facility list: 12 facilities (/Users/shirindehghannezhad/Documents/Data-Center/data/raw/datacenters_com/eu_datacenters.urls.json)
12 already done, 0 to go (roughly 0.0 hours)
Saved 12 records to data/clean/eu_datacenters.csv and eu_datacenters.json


,country,type_of_data_center,name,owner,land_use,energy_consumption_electricity,energy_consumption_water,pue,location,city,postal_code,latitude,longitude,source_of_electricity,certifications,year_built_or_operational,year_evidence,market,source_url
0,Malta,N/A,Who We Are? Datacenters.ComSM,Us,N/A,N/A,N/A,N/A,N/A,Swieqi 2,N/A,N/A,N/A,N/A,N/A,N/A,N/A,malta/swieqi/swieqi-2,https://www.datacenters.com/about-us
1,Malta,Colocation,BMIT DCH Data Centre,BMIT Technologies,"1,200 m²",N/A,N/A,N/A,N/A,Swieqi 2,N/A,N/A,N/A,N/A,ISO 27001; ISO/IEC 27001; SOC 2 Type 1; PCI DSS,N/A,N/A,malta/swieqi/swieqi-2,https://www.datacenters.com/bmit-technologies-...
2,Malta,Colocation,BMIT SmartCity Data Centre,BMIT Technologies,N/A,N/A,N/A,N/A,N/A,Swieqi 2,N/A,N/A,N/A,N/A,ISO 27001; ISO/IEC 27001; SOC 2 Type 2; SOC 2 ...,N/A,N/A,malta/swieqi/swieqi-2,https://www.datacenters.com/bmit-technologies-...
3,Malta,Colocation,Birkirkara Data Center,GO plc,N/A,N/A,N/A,N/A,N/A,Swieqi 2,N/A,N/A,N/A,N/A,ISO 14001; ISO 27001; ISO 50001; PCI DSS,N/A,N/A,malta/swieqi/swieqi-2,https://www.datacenters.com/go-plc-birkirkara
4,Malta,Colocation,GO Marsa Data Center,GO plc,N/A,N/A,N/A,N/A,N/A,Swieqi 2,N/A,N/A,N/A,N/A,ISO 14001; ISO 27001; ISO 50001,N/A,N/A,malta/swieqi/swieqi-2,https://www.datacenters.com/go-plc-go-marsa


In [ ]:
df = build_outputs(OUT)

print(df["country"].value_counts().to_string(), "\n")

coverage = (df[FIELDS] != NA).mean().mul(100).round(1).sort_values()
coverage.rename("% filled").to_frame()

Saved 12 records to data/clean/eu_datacenters.csv and eu_datacenters.json
country
Malta    12 



,% filled
year_evidence,0.0
year_built_or_operational,0.0
energy_consumption_electricity,0.0
energy_consumption_water,0.0
pue,0.0
postal_code,0.0
latitude,0.0
longitude,0.0
source_of_electricity,0.0
land_use,25.0
